# 04 — Quiver access diagnostic & House validation 2025

## Rappel critique

Quiver est une validation externe. House reste la source officielle.

Aucune correction automatique de House via Quiver.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.quiver_client import (
    get_quiver_token, diagnose_congresstrading_access, fetch_congresstrading_pages,
    normalize_quiver_records, filter_house_year, compare_house_quiver_coverage,
    write_quiver_report,
)
from src.utils import AUDIT_DIR, EXTERNAL_QUIVER_DIR, PROCESSED_HOUSE_DIR, require_file

print("Imports OK")

Imports OK


In [2]:
YEAR = 2025
PAGE_SIZE = 500
MAX_PAGES = 45  # Pages 10-40 couvrent 2025 ; marge incluse.

if load_dotenv is not None:
    load_dotenv(ROOT / ".env")

token = get_quiver_token()
print("QUIVER_API_TOKEN present:", bool(token))

QUIVER_API_TOKEN present: True


In [3]:
diagnostic_path = AUDIT_DIR / "quiver_api_access_diagnostic.json"
diagnostic = diagnose_congresstrading_access(token=token, output_path=diagnostic_path)

print("diagnostic status:", diagnostic.get("status"))
print("accessible version:", diagnostic.get("accessible_version"))
print("Written:", diagnostic_path.relative_to(ROOT))

diagnostic status: ok
accessible version: V2
Written: data/audit/quiver_api_access_diagnostic.json


In [4]:
records = []
version = diagnostic.get("accessible_version")

if token and version:
    raw_path = EXTERNAL_QUIVER_DIR / "quiver_congress_trading_raw_sample.json"
    records = fetch_congresstrading_pages(
        token=token,
        version=version,
        page_size=PAGE_SIZE,
        max_pages=MAX_PAGES,
        output_raw_path=raw_path,
    )
    print("records fetched:", len(records))
else:
    print("Skipped: no Quiver token or no accessible version.")

records fetched: 22500


In [5]:
if records:
    quiver_df = normalize_quiver_records(records, schema_version=version)
else:
    quiver_df = pd.DataFrame()

print("quiver_df shape:", quiver_df.shape)
if not quiver_df.empty:
    display(quiver_df.head())
    display(quiver_df.columns.to_series())

quiver_df shape: (22500, 20)


,source,quiver_schema_version,name,bioguide_id,chamber,party,district,state,ticker,ticker_type,company,transaction_type,transaction_date,disclosure_date,amount_range,amount_raw,description,comments,status,raw_json
0,quiver,V2,Matthew Robert Van Epps,V000139,House,Republican,7.0,Tennessee,IBM,ST,INTERNATIONAL BUSINESS MACHINES CORPORATION CO...,Sale,2026-06-16,2026-06-17,None,"$1,001 - $15,000",NaN,NaN,NEW,"{""Ticker"": ""IBM"", ""TickerType"": ""ST"", ""Company..."
1,quiver,V2,Matthew Robert Van Epps,V000139,House,Republican,7.0,Tennessee,GEV,ST,GE VERNOVA INC. COMMON STOCK,Sale,2026-06-16,2026-06-17,None,"$1,001 - $15,000",NaN,NaN,NEW,"{""Ticker"": ""GEV"", ""TickerType"": ""ST"", ""Company..."
2,quiver,V2,Matthew Robert Van Epps,V000139,House,Republican,7.0,Tennessee,GE,ST,GE AEROSPACE COMMON STOCK,Sale,2026-06-16,2026-06-17,None,"$1,001 - $15,000",NaN,NaN,NEW,"{""Ticker"": ""GE"", ""TickerType"": ""ST"", ""Company""..."
3,quiver,V2,Matthew Robert Van Epps,V000139,House,Republican,7.0,Tennessee,XOM,ST,EXXON MOBIL CORPORATION COMMON STOCK,Sale,2026-06-16,2026-06-17,None,"$1,001 - $15,000",NaN,NaN,NEW,"{""Ticker"": ""XOM"", ""TickerType"": ""ST"", ""Company..."
4,quiver,V2,Matthew Robert Van Epps,V000139,House,Republican,7.0,Tennessee,AMZN,ST,"AMAZON.COM, INC. - COMMON STOCK",Sale,2026-06-16,2026-06-17,None,"$1,001 - $15,000",NaN,NaN,NEW,"{""Ticker"": ""AMZN"", ""TickerType"": ""ST"", ""Compan..."


source                                  source
quiver_schema_version    quiver_schema_version
name                                      name
bioguide_id                        bioguide_id
chamber                                chamber
party                                    party
district                              district
state                                    state
ticker                                  ticker
ticker_type                        ticker_type
company                                company
transaction_type              transaction_type
transaction_date              transaction_date
disclosure_date                disclosure_date
amount_range                      amount_range
amount_raw                          amount_raw
description                        description
comments                              comments
status                                  status
raw_json                              raw_json
dtype: str

In [6]:
quiver_house = filter_house_year(quiver_df, YEAR) if not quiver_df.empty else pd.DataFrame()
quiver_year_path = EXTERNAL_QUIVER_DIR / f"quiver_congress_trading_{YEAR}.csv"

if not quiver_house.empty:
    quiver_house.to_csv(quiver_year_path, index=False)
    print("Written:", quiver_year_path.relative_to(ROOT))
else:
    print("No Quiver House records for selected year in fetched pages.")

Written: data/external/quiver/quiver_congress_trading_2025.csv


In [7]:
house_year_path = PROCESSED_HOUSE_DIR / f"ptr_index_{YEAR}.csv"

if house_year_path.exists():
    house_year = pd.read_csv(house_year_path, dtype={"doc_id": str})
    print(f"house_{YEAR} shape:", house_year.shape)
    display(house_year.head())
else:
    house_year = pd.DataFrame()
    print("Missing:", house_year_path.relative_to(ROOT))

house_2025 shape: (515, 11)


,year,filing_type,filing_date_raw,filing_date,doc_id,last_name,first_name,declarant_name_raw,state_dst,pdf_url,source_xml_path
0,2025,P,9/10/2025,2025-09-10,20032062,Aderholt,Robert B.,Robert B. Aderholt,AL04,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
1,2025,P,1/16/2025,2025-01-16,20026537,Allen,Richard W.,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
2,2025,P,2/20/2025,2025-02-20,20026727,Allen,Richard W.,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
3,2025,P,3/12/2025,2025-03-12,20027927,Allen,Richard W.,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...
4,2025,P,5/15/2025,2025-05-15,20030316,Allen,Richard W.,Richard W. Allen,GA12,https://disclosures-clerk.house.gov/public_dis...,/Users/lemairealice/Downloads/Jupiter/congress...


In [8]:
metrics = None
validation_path = AUDIT_DIR / f"quiver_house_validation_{YEAR}.csv"

if not house_year.empty and not quiver_house.empty:
    validation_df, metrics = compare_house_quiver_coverage(
        house_year,
        quiver_house,
        year=YEAR,
        output_path=validation_path,
    )
    display(validation_df)
    print("Written:", validation_path.relative_to(ROOT))
else:
    print("Coverage comparison skipped.")

,metric,value
0,timestamp,2026-06-23T16:09:37+00:00
1,n_house_ptr_filings_2025,515
2,n_quiver_house_transactions_2025,12527
3,n_unique_house_declarants_2025,108
4,n_unique_quiver_declarants_2025,84
5,overlap_declarants_count,80
6,house_only_declarants_count,28
7,quiver_only_declarants_count,4
8,house_disclosure_date_min,2025-01-01 00:00:00
9,house_disclosure_date_max,2026-01-16 00:00:00


Written: data/audit/quiver_house_validation_2025.csv


In [9]:
if not house_year.empty and not quiver_house.empty:
    house_keys = set(house_year["declarant_name_raw"].dropna().str.lower().str.strip())
    quiver_keys = set(quiver_house["name"].dropna().str.lower().str.strip())
    print("House-only examples:", sorted(list(house_keys - quiver_keys))[:20])
    print("Quiver-only examples:", sorted(list(quiver_keys - house_keys))[:20])
else:
    print("Divergence examples skipped.")

House-only examples: ['adrian smith', 'ann wagner', 'august lee pfluger', 'brad sherman', 'bradley s. schneider', 'chellie pingree', 'darrell e. issa', 'david p. joyce', 'donald sternoff beyer', 'earl leroy carter', 'gus m. bilirakis', 'harold dallas rogers', 'jamie raskin', 'kathy manning', 'keith alan self', 'michael a. collins', 'michael waltz', 'neal patrick md, facs dunn', 'nicole malliotakis', 'patrick ryan']
Quiver-only examples: ['august lee pfluger ii', 'ro khanna', 'rudy c. yakym iii', 'thomas h. kean jr']


In [10]:
report_path = write_quiver_report(diagnostic, metrics=metrics, year=YEAR)
print("Written:", report_path.relative_to(ROOT))

Written: reports/house_quiver_validation_report.md


## Conclusion

**OK** si le diagnostic Quiver est écrit et les limites sont explicites.

**NEXT** — ne pas corriger House avec Quiver ; utiliser les divergences comme audit.